# Logging, structlog

*0.1 Python for GenAI · run **Setup** first*

## Setup

Settings, a configured client, and two helpers. Every cell below uses them.

In [1]:
"""Shared setup for this notebook: typed settings, configured clients, logging."""

import asyncio
import json
import logging
from concurrent.futures import ThreadPoolExecutor

from dotenv import find_dotenv
from openai import AsyncOpenAI, OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """All configuration in one validated object, read from the environment / .env."""

    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()

client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def async_client() -> AsyncOpenAI:
    """A fresh async client per event loop (async clients are bound to the loop they run in)."""
    return AsyncOpenAI(
        api_key=settings.openai_api_key.get_secret_value(),
        timeout=settings.request_timeout_seconds,
        max_retries=settings.max_retries,
    )


def run_async(coroutine):
    """Run a coroutine from a notebook (which already has an event loop). Scripts use asyncio.run()."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()


def show(title: str, value) -> None:
    """Print a labelled, formatted JSON block."""
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for noisy in ["httpx", "httpx2", "httpcore", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

print("model:", settings.openai_model, "| timeout:", settings.request_timeout_seconds, "s")

model: gpt-4o-mini | timeout: 30.0 s


### Logging

> **Problem.** At 3 a.m. a customer reports wrong answers from the last hour. The service's only output is `print` statements: no timestamps, no levels, nothing saying which module wrote a line — and the log is 90% `HTTP Request: POST …` lines from the SDK.

**Idea.** Timestamped, levelled, named log lines, with library noise turned down.

**Use when** every service.  
**Not when** notebooks and one-off scripts — `print` is fine.

```
log.info("completion.start model=gpt-4o-mini")
   ──▶ 2026-09-21 10:00:01,204 INFO    app.chat: completion.start model=gpt-4o-mini
                                ▲       ▲
                              level   who wrote it
```

**How it works.**
1. `logging.getLogger("app.chat")` returns a named logger; the name appears in every line and lets you set levels per module.
2. A handler with a formatter defines the layout: timestamp, level, name, message.
3. `log.info` / `log.warning` / `log.debug` write at three levels; the logger's level (INFO here) decides what is kept — the DEBUG line is dropped.
4. `logging.getLogger("httpx2").setLevel(WARNING)` silences the SDK's per-request HTTP line; the OpenAI SDK bundles its own httpx under that name.
5. The cell writes to a buffer so the output can be checked; a service writes to stdout and the platform collects it.

| | what happens | result |
|:--|:--|:--|
| ✗ default | SDK HTTP client at INFO | one `HTTP Request: POST …` line per call, forever |
| ✓ tuned | `getLogger("httpx2").setLevel(WARNING)` | only your events remain |

**Production code and its real output**

In [2]:
# Logging — the stdlib logger with a service-style format; library loggers turned down so SDK
# HTTP chatter does not flood production logs.
import io
import time

buffer = io.StringIO()
handler = logging.StreamHandler(buffer)
handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)-7s %(name)s: %(message)s"))
log = logging.getLogger("app.chat")
log.setLevel(logging.INFO)
log.addHandler(handler)

started = time.perf_counter()
log.info("completion.start model=%s", settings.openai_model)
response = client.chat.completions.create(
    model=settings.openai_model,
    messages=[{"role": "user", "content": "Say hi in one word."}],
    temperature=0,
)
log.info(
    "completion.end seconds=%.2f tokens=%d",
    time.perf_counter() - started,
    response.usage.total_tokens,
)
log.warning("budget.near_limit spent_usd=%.4f limit_usd=%.4f", 0.048, 0.05)
log.debug("hidden at INFO level")
log.removeHandler(handler)

print(buffer.getvalue())
assert buffer.getvalue().count("\n") == 3 and "HTTP Request" not in buffer.getvalue()

INFO app.chat: completion.start model=gpt-4o-mini


INFO app.chat: completion.end seconds=0.73 tokens=15


WARNING app.chat: budget.near_limit spent_usd=0.0480 limit_usd=0.0500


2026-09-21 10:11:54,038 INFO    app.chat: completion.start model=gpt-4o-mini
2026-09-21 10:11:54,770 INFO    app.chat: completion.end seconds=0.73 tokens=15
2026-09-21 10:11:54,771 WARNING app.chat: budget.near_limit spent_usd=0.0480 limit_usd=0.0500



**What the output shows.** Three lines: start, end with timing and tokens, and a warning — each with a timestamp, level and logger name; the DEBUG line and the SDK's HTTP chatter are absent.

**In practice**
- **configure once** — at the entry point, never inside libraries; INFO in production, DEBUG locally.
- **httpx2** — the OpenAI SDK's HTTP client logs under `httpx2`, not `httpx`; silence the right name or the noise stays.
- **no secrets** — keys, tokens and full prompts stay out of logs by default; add redaction if prompts must be logged.
- **levels mean something** — WARNING is "odd but handled", ERROR is "someone should look"; if everything is ERROR, alerts are noise.
- **stdout** — write to stdout and let the platform ship it; files on a container disappear with the container.

**Alternatives** — structlog (next item — structured JSON events) · loguru (simpler API, less standard)

**Terms** — *level*: DEBUG · INFO · WARNING · ERROR — how important a line is · *logger name*: which part of the program wrote it


### structlog

> **Problem.** A million log lines a day. Someone asks "which users had requests slower than two seconds this morning?" With plain sentences the answer is a regex, a spreadsheet and an afternoon — and the request id that would tie the lines together is missing from half of them.

**Idea.** Log events as JSON records with fields, and bind request context once so every line carries it.

**Use when** logs go to a search pipeline (Datadog, Loki, CloudWatch).  
**Not when** a single developer reading a terminal.

```
plain       done in 0.42s tokens=15                                      ──▶ grep
structured  {"event": "completion.end", "seconds": 0.42, "tokens": 15,
             "request_id": "a1b2", "user_id": "u-42"}                      ──▶ filter · group · alert
```

**How it works.**
1. `structlog.configure(processors=[...])` sets the pipeline each event passes through: merge bound context, add the level, add a timestamp, render as JSON.
2. `bind_contextvars(request_id=..., user_id=..., model=...)` stores context for the current request; the merge processor attaches it to every event automatically.
3. `log.info("completion.end", seconds=0.42, tokens=15)` is an event name plus named fields — never a formatted sentence.
4. Each line is one JSON object; the log pipeline indexes the fields, so `seconds > 2 AND user_id = "u-42"` is a query, not a grep.
5. `clear_contextvars()` at the start of each request prevents one request's ids from leaking into the next in a long-lived worker.

| | what happens | result |
|:--|:--|:--|
| ✗ | "slow requests for u-42?" on plain lines | grep and guess |
| ✓ | `seconds > 2 AND user_id = "u-42"` | one query |

**Production code and its real output**

In [3]:
# structlog — JSON events with request-scoped context, the format log pipelines index.
# request_id is bound once and appears on every line of that request.
import io
import time
import uuid

import structlog

buffer = io.StringIO()
structlog.configure(
    processors=[
        structlog.contextvars.merge_contextvars,
        structlog.processors.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.JSONRenderer(),
    ],
    logger_factory=structlog.PrintLoggerFactory(file=buffer),
)
log = structlog.get_logger("app.chat")

structlog.contextvars.clear_contextvars()
structlog.contextvars.bind_contextvars(
    request_id=uuid.uuid4().hex[:8], user_id="u-42", model=settings.openai_model
)
started = time.perf_counter()
log.info("completion.start")
response = client.chat.completions.create(
    model=settings.openai_model,
    messages=[{"role": "user", "content": "Say bye in one word."}],
    temperature=0,
)
log.info(
    "completion.end",
    seconds=round(time.perf_counter() - started, 3),
    prompt_tokens=response.usage.prompt_tokens,
    completion_tokens=response.usage.completion_tokens,
)

events = []
for line in buffer.getvalue().splitlines():
    events.append(json.loads(line))
show("events", events)
assert events[0]["request_id"] == events[1]["request_id"] and events[1]["prompt_tokens"] > 0

events
[
  {
    "event": "completion.start",
    "user_id": "u-42",
    "request_id": "a3b54dfe",
    "model": "gpt-4o-mini",
    "level": "info",
    "timestamp": "2026-09-21T14:11:54.836917Z"
  },
  {
    "seconds": 0.629,
    "prompt_tokens": 13,
    "completion_tokens": 3,
    "event": "completion.end",
    "user_id": "u-42",
    "request_id": "a3b54dfe",
    "model": "gpt-4o-mini",
    "level": "info",
    "timestamp": "2026-09-21T14:11:55.465765Z"
  }
]


**What the output shows.** Two JSON events, both carrying the same `request_id`, `user_id` and `model` without repeating them in the call; the second has timing and token counts as numeric fields.

**In practice**
- **event names** — consistent names (`completion.start`, `completion.end`, `tool.call`) become dashboards and alerts; free-text messages cannot.
- **clear between requests** — workers reuse threads and tasks; unbound context from the previous request is a classic source of wrong `user_id`s in logs.
- **numbers as numbers** — `seconds=0.42`, not `seconds="0.42s"`, or the pipeline cannot compare or graph them.
- **no payloads** — full prompts and answers in logs cost money, leak personal data and bloat indexes; log ids and sizes, store bodies elsewhere if required.
- **bounded cardinality** — a field with a new value per request (`request_id`) is fine; a new *key* per request breaks indexing.

**Alternatives** — stdlib logging + a JSON formatter (same result, more wiring) · OpenTelemetry logs (when traces and logs must correlate)

**Terms** — *structured log*: a record with named fields · *contextvars*: values bound once per request, attached automatically
